# 10.9 Polars for Fast Data Analysis

Polars is a DataFrame library designed for fast, expressive data processing. Its expression system describes **what should happen to whole columns**. Polars can optimize those expressions, especially when we use the lazy API.

## What you will learn

1. Create and inspect a Polars DataFrame.
2. Select, calculate, filter, and sort with expressions.
3. Group rows and calculate summaries.
4. Handle missing values.
5. Join related tables.
6. Build and execute a lazy query.

## Installation

Run the next command once if Polars is missing. `%pip` targets the notebook's Python environment. Remove the `#` only when installation is needed.

In [ ]:
# %pip install polars

In [ ]:
# The short name pl is the standard Polars convention.
import polars as pl

# Recording a version helps another learner reproduce the notebook.
print("Polars version:", pl.__version__)

## 1. Create and inspect a DataFrame

A DataFrame is a rectangular table. Every column has a name and data type. Unlike Pandas, Polars normally does not use a row index.

In [ ]:
# Each dictionary key becomes a column in the sales table.
sales = pl.DataFrame(
    {
        "order_id": [101, 102, 103, 104, 105, 106],
        "city": ["Delhi", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi"],
        "product": ["Book", "Pen", "Bag", "Book", "Bag", "Pen"],
        "quantity": [2, 10, 1, 3, 2, 5],
        "unit_price": [350.0, 20.0, 900.0, 350.0, 900.0, 20.0],
        "rating": [5, 4, None, 3, 5, None],
    }
)

# Head shows the first rows; schema maps every column to its type.
print(sales.head(3))
print("Shape (rows, columns):", sales.shape)
print("Schema:", sales.schema)

## 2. Expressions: select and create columns

`pl.col("quantity")` is an expression meaning “use the quantity column.” An expression describes work without writing a Python loop over rows. `select` chooses output columns; `with_columns` keeps existing columns and adds or replaces columns.

In [ ]:
# Select two columns and give one a clearer display name.
small_view = sales.select(
    pl.col("order_id"),
    pl.col("product").alias("item"),
)
# Calculate revenue for every row using column expressions.
sales_with_revenue = sales.with_columns(
    (pl.col("quantity") * pl.col("unit_price")).alias("revenue")
)

# Display both results so the difference is easy to see.
print(small_view)
sales_with_revenue

## 3. Filter and sort

Filtering keeps rows where an expression is true. Parentheses around conditions make the logic explicit. Use `&` for AND, `|` for OR, and `~` for NOT.

In [ ]:
# Keep Delhi orders whose revenue is at least 500 rupees.
important_delhi_orders = (
    sales_with_revenue
    .filter(
        (pl.col("city") == "Delhi")
        & (pl.col("revenue") >= 500)
    )
    .sort("revenue", descending=True)
)

# The highest-revenue matching order appears first.
important_delhi_orders

## 4. Group and aggregate

`group_by` gathers rows with the same key. `agg` then computes one or more summaries for every group. Aliases give calculated columns helpful names.

In [ ]:
# Group orders by city and calculate several business metrics.
city_summary = (
    sales_with_revenue
    .group_by("city")
    .agg(
        pl.len().alias("orders"),
        pl.col("quantity").sum().alias("items_sold"),
        pl.col("revenue").sum().alias("total_revenue"),
        pl.col("rating").mean().round(2).alias("average_rating"),
    )
    .sort("total_revenue", descending=True)
)

# One output row now represents one city instead of one order.
city_summary

## 5. Missing values

Polars uses `null` for missing values. First count them; then decide whether to fill, remove, or investigate them. Never fill missing values without understanding what “missing” means in the real problem.

In [ ]:
# Null count reports missing values in every column.
missing_counts = sales.null_count()
# Calculate the median from ratings that are present.
median_rating = sales.select(pl.col("rating").median()).item()
# Fill only the rating column and preserve all other columns.
filled_sales = sales.with_columns(
    pl.col("rating").fill_null(median_rating)
)

# Compare the missing-value report with the filled result.
print(missing_counts)
filled_sales

## 6. Join related tables

A join combines tables using a shared key. A left join keeps every row from the left table and adds matching values from the right table. Always check that key columns have the same meaning and compatible types.

In [ ]:
# This lookup table adds a region to each city.
city_regions = pl.DataFrame(
    {
        "city": ["Delhi", "Mumbai", "Pune"],
        "region": ["North", "West", "West"],
    }
)
# The on parameter names the shared key; how chooses the join behaviour.
sales_with_regions = sales_with_revenue.join(
    city_regions,
    on="city",
    how="left",
)

# Every sales row is preserved and now has a matching region.
sales_with_regions

## 7. Lazy queries

Eager operations calculate immediately. A `LazyFrame` builds a query plan first. Polars can optimize that plan and executes it only when we call `collect()`. Lazy mode is especially useful for larger files and multi-step pipelines.

In [ ]:
# Lazy changes the eager DataFrame into a deferred LazyFrame.
revenue_query = (
    sales.lazy()
    .with_columns(
        (pl.col("quantity") * pl.col("unit_price")).alias("revenue")
    )
    .filter(pl.col("revenue") >= 500)
    .select("order_id", "city", "product", "revenue")
    .sort("revenue", descending=True)
)
# Explain displays the optimized plan without returning final rows.
print(revenue_query.explain())
# Collect executes the complete optimized plan and returns a DataFrame.
large_orders = revenue_query.collect()
large_orders

## Polars and Pandas: simple mental map

| Goal | Pandas | Polars |
| --- | --- | --- |
| Select columns | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filter rows | `df[df["a"] > 5]` | `df.filter(pl.col("a") > 5)` |
| New column | `df["c"] = ...` | `df.with_columns(...alias("c"))` |
| Group | `df.groupby("a")` | `df.group_by("a")` |
| Missing count | `df.isna().sum()` | `df.null_count()` |

Polars expressions may feel unusual initially, but they make transformations composable and optimizable.

## Summary and practice

- Use `select` to choose output columns.
- Use `with_columns` to add or replace calculated columns.
- Build filters with `pl.col(...)` expressions.
- Combine `group_by` with `agg` for summaries.
- Inspect nulls before deciding how to handle them.
- Use joins to combine related tables.
- Use lazy queries for optimizable data pipelines.

### Practice

1. Add a `discount` column and calculate revenue after discount.
2. Find products with a quantity greater than two.
3. Group by product and calculate total revenue.
4. Replace missing ratings with the city average instead of the global median.
5. Add a region that has no sales and compare left and inner joins.
6. Rewrite your group summary as a lazy query and inspect its plan.